### Installation

In [5]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2

### Unsloth

In [6]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit", # Llama 3.2 vision support
    "unsloth/Llama-3.2-11B-Vision-bnb-4bit",
    "unsloth/Llama-3.2-90B-Vision-Instruct-bnb-4bit", # Can fit in a 80GB card!
    "unsloth/Llama-3.2-90B-Vision-bnb-4bit",

    "unsloth/Pixtral-12B-2409-bnb-4bit",              # Pixtral fits in 16GB!
    "unsloth/Pixtral-12B-Base-2409-bnb-4bit",         # Pixtral base model

    "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",          # Qwen2 VL support
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-72B-Instruct-bnb-4bit",

    "unsloth/llava-v1.6-mistral-7b-hf-bnb-4bit",      # Any Llava variant works!
    "unsloth/llava-1.5-7b-hf-bnb-4bit",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-4B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.30G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/213 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/782 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

In [7]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [4]:
import json
import random
from pathlib import Path
from PIL import Image

SEED = 42
random.seed(SEED)

DATASET_PATHS = [
    "/mnt/origin-assn/dataset1_yolo_seg",
    "/mnt/origin-assn/dataset2_yolo"
]
OUTPUT_FILE = "/mnt/origin-assn/qwen_vl_train.jsonl"

CLASS_MAP = {
    0: ['drywall-join',  "segment taping area", "segment joint/tape", "segment drywall seam"],
    1: ['crack', "segment crack", "segment wall crack", "wall crack"]
}


def convert_yolo_seg_to_qwen_vl(dataset_paths, output_file):
    lines_written = 0

    with open(output_file, 'w') as f_out:
        for dataset_path in dataset_paths:
            dataset_path = Path(dataset_path)

            for split in ['train', 'valid', 'test']:
                img_dir = dataset_path / split / 'images'
                lbl_dir = dataset_path / split / 'labels'

                if not img_dir.exists():
                    continue

                print(f"Processing {img_dir}...")

                for img_file in sorted(img_dir.glob('*.*')):
                    if img_file.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                        continue

                    lbl_file = lbl_dir / f"{img_file.stem}.txt"
                    if not lbl_file.exists():
                        continue

                    try:
                        with Image.open(img_file) as im:
                            w, h = im.size
                    except Exception:
                        continue

                    annotations_by_class = {}

                    with open(lbl_file, 'r') as f_in:
                        for line in f_in:
                            parts = line.strip().split()
                            # Segmentation: class_id + >=6 coords (3+ xy pairs)
                            if len(parts) < 7:
                                continue

                            cls_id = int(parts[0])
                            coords = list(map(float, parts[1:]))

                            # Denormalise polygon: [x1,y1, x2,y2, ...]
                            polygon = []
                            for i in range(0, len(coords) - 1, 2):
                                px = round(coords[i]     * w)
                                py = round(coords[i + 1] * h)
                                polygon.append([px, py])

                            if cls_id not in annotations_by_class:
                                annotations_by_class[cls_id] = []
                            annotations_by_class[cls_id].append(polygon)

                    for cls_id, polygons in annotations_by_class.items():
                        prompts = CLASS_MAP.get(cls_id, [f"class_{cls_id}"])[1:]  # skip name at index 0
                        prompt_text = random.choice(prompts)

                        prompt_text = f"{prompt_text} in this image and output the segmentation coordinates (polygon) in JSON format."

                        if len(polygons) == 1:
                            response = json.dumps({"polygon": polygons[0]})
                        else:
                            response = json.dumps({"polygons": polygons})

                        entry = {
                            "messages": [
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "text",  "text":  prompt_text},
                                        {"type": "image", "image": str(img_file)}
                                    ]
                                },
                                {
                                    "role": "assistant",
                                    "content": [
                                        {"type": "text", "text": response}
                                    ]
                                }
                            ]
                        }

                        f_out.write(json.dumps(entry) + '\n')
                        lines_written += 1

    print(f"Conversion complete. Total entries: {lines_written}")


convert_yolo_seg_to_qwen_vl(DATASET_PATHS, OUTPUT_FILE)

Processing /mnt/origin-assn/dataset1_yolo_seg/train/images...
Processing /mnt/origin-assn/dataset1_yolo_seg/valid/images...
Processing /mnt/origin-assn/dataset2_yolo/train/images...
Processing /mnt/origin-assn/dataset2_yolo/valid/images...
Processing /mnt/origin-assn/dataset2_yolo/test/images...
Conversion complete. Total entries: 1737


To format the dataset, all vision finetuning tasks should be formatted as follows:

```python
[
{ "role": "user",
  "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
},
{ "role": "assistant",
  "content": [{"type": "text",  "text": A} ]
},
]
```

In [8]:
import json

with open("/mnt/origin-assn/qwen_vl_train.jsonl") as f:
    converted_dataset = [json.loads(line) for line in f]

We look at how the conversations are structured for the first example:

In [9]:
converted_dataset[0]

{'messages': [{'role': 'user',
   'content': [{'type': 'text',
     'text': 'segment drywall seam in this image and output the segmentation coordinates (polygon) in JSON format.'},
    {'type': 'image',
     'image': '/mnt/origin-assn/dataset1_yolo_seg/train/images/2000x1500_0_resized_jpg.rf.0dd5a8210e3178cb5374e1bd32333ff1.jpg'}]},
  {'role': 'assistant',
   'content': [{'type': 'text',
     'text': '{"polygon": [[288, 0], [324, 0], [324, 640], [288, 640]]}'}]}]}

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [10]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # max_steps = 80,
        num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [11]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
4.189 GB of memory reserved.


In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,737 | Num Epochs = 1 | Total steps = 109
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 39,321,600 of 4,477,137,408 (0.88% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.509600
2,1.199400
3,1.127700
4,1.142400
5,1.387700
6,1.098700
7,0.997800
8,0.941900
9,1.010200
10,0.799400


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
!pip install opencv-python pillow -q
import json, re, cv2, numpy as np
from pathlib import Path
from PIL import Image
import torch

CLASSES = {0: "drywall taping area", 1: "wall crack"}
PROMPTS = {0: "segment taping area", 1: "segment crack"}

def parse_polygon(text):
    """Extract first polygon array from model JSON response."""
    try:
        data = json.loads(re.search(r'\{.*\}', text, re.DOTALL).group())
        pts  = data.get("polygon") or data.get("polygons", [[]])[0]
        return np.array(pts, dtype=np.int32)
    except Exception:
        return None

def rasterise(polygon, h, w):
    """Fill closed polygon into binary mask."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if polygon is not None and len(polygon) >= 3:
        cv2.fillPoly(mask, [polygon.reshape(-1, 1, 2)], 1)
    return mask

def compute_metrics(pred, gt):
    pred, gt     = pred.astype(bool), gt.astype(bool)
    intersection = (pred & gt).sum()
    union        = (pred | gt).sum()
    iou  = intersection / union                        if union  > 0 else float("nan")
    dice = 2 * intersection / (pred.sum() + gt.sum()) if (pred.sum() + gt.sum()) > 0 else float("nan")
    return iou, dice

def load_pairs(dataset_path, split="valid"):
    img_dir = Path(dataset_path) / split / "images"
    lbl_dir = Path(dataset_path) / split / "labels"
    return [(p, lbl_dir / f"{p.stem}.txt") for p in sorted(img_dir.glob("*.jpg"))
            if (lbl_dir / f"{p.stem}.txt").exists()]

def eval_split(pairs, cls_id, model, tokenizer):
    prompt_text = f"{PROMPTS[cls_id]} in this image and output the segmentation coordinates (polygon) in JSON format."
    ious, dices = [], []

    for img_path, lbl_path in pairs:
        img  = Image.open(img_path)
        w, h = img.size

        # Ground truth
        gt_mask = np.zeros((h, w), dtype=np.uint8)
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 7 or int(parts[0]) != cls_id:
                    continue
                coords = np.array(parts[1:], dtype=np.float32)
                pts    = (coords.reshape(-1, 2) * np.array([w, h])).astype(np.int32)
                cv2.fillPoly(gt_mask, [pts], 1)

        # Inference — unsloth FastVisionModel tokenizer API
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_text}
        ]}]
        input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        inputs     = tokenizer(img, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
        out_ids    = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=1.5, min_p=0.1)
        response   = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

        polygon   = parse_polygon(response)
        pred_mask = rasterise(polygon, h, w)

        iou, dice = compute_metrics(pred_mask, gt_mask)
        ious.append(iou); dices.append(dice)

    return np.nanmean(ious), np.nanmean(dices)

from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "/root/drywall-segment-vl", # YOUR MODEL YOU USED FOR TRAINING
    load_in_4bit = True, # Set to False for 16bit LoRA
)
FastVisionModel.for_inference(model) # Enable for inference!

datasets = {
    "dataset1 (taping)": ("/mnt/origin-assn/dataset1_yolo_seg", 0),
    "dataset2 (cracks)": ("/mnt/origin-assn/dataset2_yolo",     1),
}

print(f"\n{'Split':<25} {'mIoU':>8} {'Dice':>8}")
print("-" * 43)
all_iou, all_dice = [], []
for name, (path, cls_id) in datasets.items():
    pairs       = load_pairs(path, split="valid")
    print(f"Evaluating {name} ({len(pairs)} images)...", end=" ", flush=True)
    miou, mdice = eval_split(pairs, cls_id, model, tokenizer)
    all_iou.append(miou); all_dice.append(mdice)
    print(f"{name} {miou} {mdice}")

print("-" * 43)
print(f"{'Mean':<25} {np.nanmean(all_iou):>8.4f} {np.nanmean(all_dice):>8.4f}")


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
==((====))==  Unsloth 2026.3.4: Fast Qwen3_Vl patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

Split                         mIoU     Dice
-------------------------------------------
Evaluating dataset1 (taping) (202 images)... 

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [13]:
model.save_pretrained("drywall-segment-vl")  # Local saving
tokenizer.save_pretrained("drywall-segment-vl")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

[]